# 05 — PyTorch Dynamic Quantization

> Companion to **Week 11**, Part 5 of the lecture.

## What you will do

Train a tiny MLP on the digits dataset, then **quantize** it to INT8 with
**one line of PyTorch code**, and compare:

| Variant | Size on disk | Accuracy | Speed |
|---|---|---|---|
| FP32 (original) | ? | ? | ? |
| INT8 (quantized) | ? | ? | ? |

You will fill in the `?` boxes from your own measurements.


## Step 1 — Load data and define a model


In [ ]:
import io
import time

import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

try:
    from torch.ao.quantization import quantize_dynamic
except ImportError:
    from torch.quantization import quantize_dynamic

torch.manual_seed(42)

digits = load_digits()
X = torch.tensor(digits.data / 16.0, dtype=torch.float32)
y = torch.tensor(digits.target, dtype=torch.long)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=256)


class SmallMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.net(x)


## Step 2 — Helper functions and training loop


In [ ]:
def evaluate(model):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in test_loader:
            preds = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += len(yb)
    return correct / total


def model_size_kb(model):
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    return len(buffer.getvalue()) / 1024


model = SmallMLP()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(8):
    model.train()
    for xb, yb in train_loader:
        loss = loss_fn(model(xb), yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

fp32_acc  = evaluate(model)
fp32_size = model_size_kb(model)
print(f"FP32 — accuracy: {fp32_acc:.3f}   size: {fp32_size:.1f} KB")


## Step 3 — Quantize in ONE line

This is the line of code worth remembering:

```python
quantized = quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)
```

That's it. We tell PyTorch which kinds of layers to quantize (`nn.Linear`),
and the target precision (`torch.qint8`). PyTorch handles the rest.


In [ ]:
quantized_model = quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)

int8_acc  = evaluate(quantized_model)
int8_size = model_size_kb(quantized_model)

sample_batch = X_test[:256]


def benchmark(m, runs=300):
    m.eval()
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(runs):
            m(sample_batch)
    return (time.perf_counter() - start) * 1000 / runs


fp32_ms = benchmark(model)
int8_ms = benchmark(quantized_model)

pd.DataFrame(
    [
        {"model": "fp32", "accuracy": fp32_acc, "size_kb": fp32_size, "latency_ms": fp32_ms},
        {"model": "int8", "accuracy": int8_acc, "size_kb": int8_size, "latency_ms": int8_ms},
    ]
)


### What you should see — and how to read it honestly

A typical Colab CPU run of this notebook prints something like:

```
   model    accuracy    size_kb    latency_ms
0   fp32    0.955       70.0       0.45
1   int8    0.958       22.0       2.14
```

Three things to notice:

**1. Size went down ~3×** (70 KB → 22 KB). That's the reliable win and it
matches our theory: `nn.Linear` weights now use 1 byte instead of 4.

**2. Accuracy looks *higher* for INT8.** Don't believe it. The test set has
~360 samples, so one prediction flipping is worth `±0.28%`. Re-run the whole
notebook with `torch.manual_seed(0)` vs `torch.manual_seed(7)` and the winner
will swap. The honest read is **same accuracy**.

**3. INT8 is *slower*.** Yes — on a model this small, on CPU. Dynamic
quantization adds per-call overhead (observe activations → pick scale →
quantize → matmul in INT8 → dequantize). On a `64→128→64→10` MLP the matmul
is so cheap that the overhead dominates.

> The **size win is always there**. The **speed win only shows up once the
> matmul is big enough** that scaling overhead disappears in the noise — think
> BERT-sized and above. For tiny MLPs, quantize for *size* only.

## 🧪 Your turn

1. Train for more epochs (e.g. 30 instead of 8). Does the "accuracy difference"
   between FP32 and INT8 stay the same or shrink? (It should be noise either way.)
2. Make the model **much** wider — `nn.Linear(64, 1024)` and `nn.Linear(1024, 1024)`
   in the middle. Re-time both versions. Does INT8 start winning on latency?
3. Print `quantized_model.net[0]` and look at the layer type. What is it
   actually? (Hint: it's no longer `Linear`.)
